# Combine output from multisystem runs

<!-- index:description -->
Collect OpenMC runtimes from six different computing systems. To gain familiarity with the OpenMC ecosystem, extract runtime data in three ways: from stdout logs, through `openmc.StatePoint`, and directly from HDF5 files using `h5py`.
<!-- /index:description -->

<!-- generated:structure -->
```text
Combine output from multisystem runs
├── 1. Get runtime data from stdout logs
├── 2. Get runtime data from `openmc.StatePoint`
│   ├── Find statepoint with the highest batch number
│   └── Extract data using openmc.StatePoint
├── 3. Get runtime data from h5py
└── 4. Match runtime data from stdout, StatePoint, and h5py
```
<!-- /generated:structure -->


In [1]:
import glob
from pathlib import Path
import re

import h5py
import matplotlib.pyplot as plt
import numpy as np
import openmc
import pandas as pd

plt.style.use('../workbench.mplstyle')

path = Path('analog_ce_reference/runs/analog_ce_seed_*')

## 1. Get runtime data from stdout logs

In [2]:
records = []

logfiles = sorted(glob.glob(str(path / '*.log')))
for logfile in logfiles:
    with open(logfile) as fp:
        dic = {'Seed': int(re.search(r'_seed_(\d+)\.log', logfile).group(1))}
        lines = fp.read().splitlines()
        for line in lines:
            if ' OpenMP Threads | ' in line:
                dic['OpenMP Threads'] = int(line.split()[-1])
            elif ' MPI Processes | '  in line:
                dic['MPI Processes'] = line.split()[-1]
            elif ' = ' in line and 'second' in line:
                splits = line.split('=')
                dic[splits[0].strip()] = float(splits[-1].split()[0])
    records.append(dic)
df = pd.DataFrame(records)
assert df['Seed'].nunique() == len(df)
seed_to_system = {
    1: 'workstation',
    100001: 'slurm #1',
    300003: 'server',
    400001: '6 containers',
    400007: 'wsl',
}
seed_to_system.update(dict.fromkeys(range(100003, 100013), 'slurm #1'))
seed_to_system.update(dict.fromkeys(range(200001, 200006), 'slurm #2'))
seed_to_system.update(dict.fromkeys(range(400001, 400007), '6 containers'))
df['System'] = df['Seed'].map(seed_to_system)
assert df['System'].notna().all(), 'Some seeds have no assigned system'

df.set_index('Seed', drop=True, inplace=True)

df['MPI Processes'] = df['MPI Processes'].fillna(1)
df['MPI Processes'] = df['MPI Processes'].astype(int)
df['Execution Threads'] = df.apply(lambda x: x['OpenMP Threads'] * x['MPI Processes'], axis=1)
df['Execution Threads'] = df['Execution Threads'].astype(int)
df['Rate Per Thread'] = df.apply(lambda x: x['Calculation Rate (active)'] / x['Execution Threads'], axis=1)

df.sort_values('Rate Per Thread', ascending=False)

,OpenMP Threads,Total time for initialization,Reading cross sections,Total time in simulation,Time in transport only,Time in active batches,Time accumulating tallies,Time writing statepoints,Total time for finalization,Total time elapsed,Calculation Rate (active),MPI Processes,System,Execution Threads,Rate Per Thread
Seed,,,,,,,,,,,,,,,
200005,2,2.1375,1.8603,163750.0,163050.0,163750.0,700.5000,0.187430,1.546800e-03,163750.0,11767.80,3,slurm #2,6,1961.300000
200002,2,2.4743,1.9490,161170.0,161030.0,161170.0,137.6700,0.132170,1.358400e-03,161170.0,11453.90,3,slurm #2,6,1908.983333
200004,2,2.2369,1.9696,158720.0,158250.0,158720.0,471.3700,0.184680,1.999700e-03,158720.0,11384.70,3,slurm #2,6,1897.450000
200003,2,2.4664,1.9140,162630.0,159950.0,162630.0,2672.0000,0.197440,1.432400e-03,162630.0,11357.40,3,slurm #2,6,1892.900000
200001,2,2.5193,1.9245,161320.0,159730.0,161320.0,1588.6000,0.120180,1.283000e-03,161320.0,11288.10,3,slurm #2,6,1881.350000
300003,256,4.6012,4.4712,149230.0,149220.0,149230.0,9.4617,0.119280,1.081000e-06,149230.0,410933.00,1,server,256,1605.207031
100012,2,9.4086,9.2780,156330.0,156040.0,156330.0,284.1200,0.089564,3.326900e-03,156340.0,9141.12,3,slurm #1,6,1523.520000
100004,2,6.2560,6.0971,157000.0,156600.0,157000.0,404.6800,0.060670,3.384300e-03,157010.0,8210.14,3,slurm #1,6,1368.356667
100003,2,6.2579,6.0973,157600.0,156900.0,157600.0,691.9700,0.032625,2.986000e-03,157600.0,8172.76,3,slurm #1,6,1362.126667


## 2. Get runtime data from `openmc.StatePoint`

### Find statepoint with the highest batch number

In [3]:
spfiles = sorted(glob.glob(str(path / '*_batch_*.h5')))
df_sp = pd.DataFrame(columns=['seed', 'batch'])
for spfile in spfiles:
    seed = int(re.search(r'_seed_(\d+)_batch',spfile).group(1))
    batch = int(re.search(r'_batch_(\d+)\.h5',spfile).group(1))
    df_sp.loc[spfile] = [seed, batch]

for seed, dat in df_sp.groupby('seed'):
    df.loc[seed, 'sp_maxbatch'] = dat['batch'].idxmax()

df

,OpenMP Threads,Total time for initialization,Reading cross sections,Total time in simulation,Time in transport only,Time in active batches,Time accumulating tallies,Time writing statepoints,Total time for finalization,Total time elapsed,Calculation Rate (active),MPI Processes,System,Execution Threads,Rate Per Thread,sp_maxbatch
Seed,,,,,,,,,,,,,,,,
1,36,1.4560,1.4430,215470.0,215460.0,215470.0,5.1236,0.038131,3.490000e-07,215470.0,26574.60,1,workstation,36,738.183333,analog_ce_reference/runs/analog_ce_seed_000001...
100001,2,5.6015,5.4887,164160.0,163250.0,164160.0,908.7900,0.061467,4.507500e-03,164160.0,7724.32,3,slurm #1,6,1287.386667,analog_ce_reference/runs/analog_ce_seed_100001...
100003,2,6.2579,6.0973,157600.0,156900.0,157600.0,691.9700,0.032625,2.986000e-03,157600.0,8172.76,3,slurm #1,6,1362.126667,analog_ce_reference/runs/analog_ce_seed_100003...
100004,2,6.2560,6.0971,157000.0,156600.0,157000.0,404.6800,0.060670,3.384300e-03,157010.0,8210.14,3,slurm #1,6,1368.356667,analog_ce_reference/runs/analog_ce_seed_100004...
100005,2,6.2625,6.0971,157780.0,157680.0,157780.0,99.3960,0.038303,2.967000e-03,157790.0,8163.23,3,slurm #1,6,1360.538333,analog_ce_reference/runs/analog_ce_seed_100005...
100006,2,4.5797,4.5336,158710.0,157620.0,158710.0,1093.0000,0.044458,3.118100e-03,158720.0,8058.58,3,slurm #1,6,1343.096667,analog_ce_reference/runs/analog_ce_seed_100006...
100007,2,4.6585,4.6208,157180.0,156360.0,157180.0,825.0200,0.034190,2.992000e-03,157190.0,8041.51,3,slurm #1,6,1340.251667,analog_ce_reference/runs/analog_ce_seed_100007...
100008,2,6.2571,6.0968,159690.0,158560.0,159690.0,1135.7000,0.051992,3.091800e-03,159700.0,8065.44,3,slurm #1,6,1344.240000,analog_ce_reference/runs/analog_ce_seed_100008...
100009,2,4.5428,4.4834,157350.0,156760.0,157350.0,592.1300,0.032872,3.057100e-03,157360.0,8051.98,3,slurm #1,6,1341.996667,analog_ce_reference/runs/analog_ce_seed_100009...


### Extract data using openmc.StatePoint

In [4]:
orig_columns = df.columns
for seed, statepoint in df['sp_maxbatch'].items():
    with openmc.StatePoint(statepoint) as sp:
        for name, seconds in sp.runtime.items():
            assert name not in orig_columns
            df.loc[seed, name] = seconds
print('Columns added from openmc.StatePoint: ', set(df.columns).difference(orig_columns))
df

Columns added from openmc.StatePoint:  {'accumulating tallies', 'transport', 'active batches', 'total initialization', 'total', 'reading cross sections', 'writing statepoints', 'simulation'}


,OpenMP Threads,Total time for initialization,Reading cross sections,Total time in simulation,Time in transport only,Time in active batches,Time accumulating tallies,Time writing statepoints,Total time for finalization,Total time elapsed,...,Rate Per Thread,sp_maxbatch,accumulating tallies,active batches,reading cross sections,simulation,total,total initialization,transport,writing statepoints
Seed,,,,,,,,,,,,,,,,,,,,,
1,36,1.4560,1.4430,215470.0,215460.0,215470.0,5.1236,0.038131,3.490000e-07,215470.0,...,738.183333,analog_ce_reference/runs/analog_ce_seed_000001...,5.123560,215469.263247,1.443023,215469.263105,215470.726888,1.455984,215463.795642,0.037650
100001,2,5.6015,5.4887,164160.0,163250.0,164160.0,908.7900,0.061467,4.507500e-03,164160.0,...,1287.386667,analog_ce_reference/runs/analog_ce_seed_100001...,908.787900,164156.928875,5.488674,164156.928810,164162.540311,5.601466,163247.688515,0.054905
100003,2,6.2579,6.0973,157600.0,156900.0,157600.0,691.9700,0.032625,2.986000e-03,157600.0,...,1362.126667,analog_ce_reference/runs/analog_ce_seed_100003...,691.969989,157596.699208,6.097296,157596.699150,157602.967051,6.257942,156904.382121,0.026216
100004,2,6.2560,6.0971,157000.0,156600.0,157000.0,404.6800,0.060670,3.384300e-03,157010.0,...,1368.356667,analog_ce_reference/runs/analog_ce_seed_100004...,404.680431,157000.992478,6.097078,157000.992415,157007.259538,6.255970,156595.892786,0.054280
100005,2,6.2625,6.0971,157780.0,157680.0,157780.0,99.3960,0.038303,2.967000e-03,157790.0,...,1360.538333,analog_ce_reference/runs/analog_ce_seed_100005...,99.396097,157780.765446,6.097127,157780.765393,157787.037707,6.262496,157680.964002,0.031849
100006,2,4.5797,4.5336,158710.0,157620.0,158710.0,1093.0000,0.044458,3.118100e-03,158720.0,...,1343.096667,analog_ce_reference/runs/analog_ce_seed_100006...,1093.002309,158712.805185,4.533635,158712.805132,158717.395875,4.579695,157619.446410,0.038317
100007,2,4.6585,4.6208,157180.0,156360.0,157180.0,825.0200,0.034190,2.992000e-03,157190.0,...,1340.251667,analog_ce_reference/runs/analog_ce_seed_100007...,825.017944,157184.438865,4.620821,157184.438802,157189.107160,4.658517,156359.073909,0.027413
100008,2,6.2571,6.0968,159690.0,158560.0,159690.0,1135.7000,0.051992,3.091800e-03,159700.0,...,1344.240000,analog_ce_reference/runs/analog_ce_seed_100008...,1135.725833,159693.703924,6.096769,159693.703872,159699.970792,6.257077,158557.614251,0.045581
100009,2,4.5428,4.4834,157350.0,156760.0,157350.0,592.1300,0.032872,3.057100e-03,157360.0,...,1341.996667,analog_ce_reference/runs/analog_ce_seed_100009...,592.131750,157352.526989,4.483423,157352.526929,157357.080070,4.542782,156760.046462,0.026488


## 3. Get runtime data from h5py

In [5]:
cols = set(df.columns)

for seed, statepoint in df['sp_maxbatch'].items():
    with h5py.File(statepoint, 'r') as f:
        for name, dataset in f['runtime'].items():
            df.loc[seed, f'h5py: {name}'] = dataset[()]

print('Columns added from h5py:', set(df.columns) - cols)
df


Columns added from h5py: {'h5py: writing statepoints', 'h5py: total', 'h5py: simulation', 'h5py: active batches', 'h5py: transport', 'h5py: reading cross sections', 'h5py: accumulating tallies', 'h5py: total initialization'}


,OpenMP Threads,Total time for initialization,Reading cross sections,Total time in simulation,Time in transport only,Time in active batches,Time accumulating tallies,Time writing statepoints,Total time for finalization,Total time elapsed,...,transport,writing statepoints,h5py: accumulating tallies,h5py: active batches,h5py: reading cross sections,h5py: simulation,h5py: total,h5py: total initialization,h5py: transport,h5py: writing statepoints
Seed,,,,,,,,,,,,,,,,,,,,,
1,36,1.4560,1.4430,215470.0,215460.0,215470.0,5.1236,0.038131,3.490000e-07,215470.0,...,215463.795642,0.037650,5.123560,215469.263247,1.443023,215469.263105,215470.726888,1.455984,215463.795642,0.037650
100001,2,5.6015,5.4887,164160.0,163250.0,164160.0,908.7900,0.061467,4.507500e-03,164160.0,...,163247.688515,0.054905,908.787900,164156.928875,5.488674,164156.928810,164162.540311,5.601466,163247.688515,0.054905
100003,2,6.2579,6.0973,157600.0,156900.0,157600.0,691.9700,0.032625,2.986000e-03,157600.0,...,156904.382121,0.026216,691.969989,157596.699208,6.097296,157596.699150,157602.967051,6.257942,156904.382121,0.026216
100004,2,6.2560,6.0971,157000.0,156600.0,157000.0,404.6800,0.060670,3.384300e-03,157010.0,...,156595.892786,0.054280,404.680431,157000.992478,6.097078,157000.992415,157007.259538,6.255970,156595.892786,0.054280
100005,2,6.2625,6.0971,157780.0,157680.0,157780.0,99.3960,0.038303,2.967000e-03,157790.0,...,157680.964002,0.031849,99.396097,157780.765446,6.097127,157780.765393,157787.037707,6.262496,157680.964002,0.031849
100006,2,4.5797,4.5336,158710.0,157620.0,158710.0,1093.0000,0.044458,3.118100e-03,158720.0,...,157619.446410,0.038317,1093.002309,158712.805185,4.533635,158712.805132,158717.395875,4.579695,157619.446410,0.038317
100007,2,4.6585,4.6208,157180.0,156360.0,157180.0,825.0200,0.034190,2.992000e-03,157190.0,...,156359.073909,0.027413,825.017944,157184.438865,4.620821,157184.438802,157189.107160,4.658517,156359.073909,0.027413
100008,2,6.2571,6.0968,159690.0,158560.0,159690.0,1135.7000,0.051992,3.091800e-03,159700.0,...,158557.614251,0.045581,1135.725833,159693.703924,6.096769,159693.703872,159699.970792,6.257077,158557.614251,0.045581
100009,2,4.5428,4.4834,157350.0,156760.0,157350.0,592.1300,0.032872,3.057100e-03,157360.0,...,156760.046462,0.026488,592.131750,157352.526989,4.483423,157352.526929,157357.080070,4.542782,156760.046462,0.026488


## 4. Match runtime data from stdout, StatePoint, and h5py


In [6]:
pairs = {
    'Total time for initialization': 'total initialization',
    'Reading cross sections': 'reading cross sections',
    'Total time in simulation': 'simulation',
    'Time in transport only': 'transport',
    'Time in active batches': 'active batches',
    'Time accumulating tallies': 'accumulating tallies',
    'Total time elapsed': 'total',
}

for log_col, sp_col in pairs.items():
    h5_col = f'h5py: {sp_col}'
    assert np.allclose(df[log_col], df[sp_col], rtol=5e-5, atol=1e-4), \
        f'Mismatch: {log_col} / {sp_col}'
    assert np.allclose(df[log_col], df[h5_col], rtol=5e-5, atol=1e-4), \
        f'Mismatch: {log_col} / {h5_col}'
    assert np.array_equal(df[sp_col], df[h5_col]), \
        f'Mismatch: {sp_col} / {h5_col}'

All 7 runtime fields match across stdout, StatePoint, and h5py.
